# 5. EDP classiques : Poisson, chaleur, ondes

**Objectif** : simuler numériquement les trois EDP classiques, vérifier les ordres de convergence
et observer le comportement qualitatif des solutions.

| EDP | Équation | Type |
|-----|---------|------|
| Poisson | $-\Delta u = f$ | Elliptique |
| Chaleur | $\partial_t u - \Delta u = 0$ | Parabolique |
| Ondes  | $\partial_{tt} u - c^2 \Delta u = 0$ | Hyperbolique |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# ══════════════════════════════════════════════════════════════════════════════
# A. ÉQUATION DE POISSON 2D  : -Δu = f sur [0,1]²,  u = 0 sur ∂Ω
# Méthode : différences finies centrées O(h²)
# Solution de test : u(x,y) = sin(πx)sin(πy),  f = 2π²u
# ══════════════════════════════════════════════════════════════════════════════

def poisson_2d_FD(N=50):
    h     = 1.0 / (N + 1)
    xi    = np.linspace(h, 1-h, N)
    X, Y  = np.meshgrid(xi, xi)

    # Second membre f = 2π² sin(πx)sin(πy)
    F_mat = 2 * np.pi**2 * np.sin(np.pi*X) * np.sin(np.pi*Y)
    f_vec = F_mat.ravel()

    # Matrice laplacien 2D par bloc : -Δ_h = (1/h²)(T⊗I + I⊗T)
    # T = tridiag(-1, 2, -1) de taille N×N
    from scipy.sparse import diags, kron, eye
    from scipy.sparse.linalg import spsolve
    T   = diags([-1, 2, -1], [-1, 0, 1], shape=(N, N))
    I   = eye(N)
    A   = (1/h**2) * (kron(I, T) + kron(T, I))
    u   = spsolve(A, f_vec).reshape(N, N)

    u_exact = np.sin(np.pi*X) * np.sin(np.pi*Y)
    err     = np.max(np.abs(u - u_exact))
    return X, Y, u, u_exact, err, h

X, Y, u, u_ex, err, h = poisson_2d_FD(N=40)
print(f'Poisson 2D  — Erreur max ||u_h - u||_inf = {err:.2e}  (h={h:.3f}, h²={h**2:.2e})')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, data, title in zip(axes, [u, np.abs(u - u_ex)],
                           ['Solution numérique $u_h$', 'Erreur $|u_h - u_{exact}|$']):
    im = ax.contourf(X, Y, data, 40, cmap='viridis')
    plt.colorbar(im, ax=ax)
    ax.set_title(title); ax.set_aspect('equal')
plt.suptitle('Poisson 2D : $-\\Delta u = 2\\pi^2 \\sin(\\pi x)\\sin(\\pi y)$', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# B. ÉQUATION DE LA CHALEUR 1D : ∂_t u = ∂_xx u,  u(0,t)=u(1,t)=0
# Condition initiale : u(x,0) = sin(πx)
# Solution exacte   : u(x,t) = exp(-π²t) sin(πx)
# Schéma : Euler implicite (inconditionnellement stable)
# ══════════════════════════════════════════════════════════════════════════════

def chaleur_euler_implicite(N=80, T=0.5, dt=0.005):
    h  = 1.0 / (N + 1)
    x  = np.linspace(h, 1-h, N)
    nt = int(T / dt)
    r  = dt / h**2  # nombre de Fourier

    # Matrice (I + r*L), L = tridiag(-1,2,-1)
    from scipy.linalg import solve_banded
    diag  = (1 + 2*r) * np.ones(N)
    upper = -r * np.ones(N-1)
    lower = -r * np.ones(N-1)
    ab    = np.zeros((3, N))
    ab[0, 1:]  = upper
    ab[1, :]   = diag
    ab[2, :-1] = lower

    u  = np.sin(np.pi * x)   # CI
    snapshots = [(0.0, u.copy())]

    for k in range(nt):
        u = solve_banded((1, 1), ab, u)
        if k in [int(0.05/dt)-1, int(0.2/dt)-1, nt-1]:
            snapshots.append(((k+1)*dt, u.copy()))

    return x, snapshots

x, snaps = chaleur_euler_implicite()
u_exact_chaleur = lambda x, t: np.exp(-np.pi**2 * t) * np.sin(np.pi * x)

plt.figure(figsize=(9, 4))
colors = ['steelblue', 'orange', 'green', 'tomato']
for (t, u), col in zip(snaps, colors):
    plt.plot(x, u, '-', color=col, label=f't={t:.3f} (numérique)')
    plt.plot(x, u_exact_chaleur(x, t), '--', color=col, alpha=0.6,
             label=f't={t:.3f} (exact)' if t == 0 else None)
plt.title('Équation de la chaleur — Euler implicite')
plt.xlabel('x'); plt.legend(fontsize=8); plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Vérification erreur au temps final
t_f, u_f = snaps[-1]
err_chaleur = np.max(np.abs(u_f - u_exact_chaleur(x, t_f)))
print(f'Erreur max à t={t_f} : {err_chaleur:.2e}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# C. ÉQUATION DES ONDES 1D : ∂_tt u = c² ∂_xx u
# Condition initiale : u(x,0) = sin(πx),  ∂_t u(x,0) = 0
# Solution exacte   : u(x,t) = cos(c π t) sin(πx)
# Schéma : Newmark (beta=1/4, gamma=1/2) — inconditionnellement stable
# ══════════════════════════════════════════════════════════════════════════════

def ondes_newmark(c=1.0, N=80, T=2.0, dt=0.02, beta=0.25, gamma=0.5):
    h    = 1.0 / (N + 1)
    x    = np.linspace(h, 1-h, N)
    nt   = int(T / dt)

    # Matrice raideur K = c²/h² * tridiag(-1,2,-1)
    k    = c**2 / h**2
    diag = 2*k * np.ones(N)
    offd = -k  * np.ones(N-1)
    K    = np.diag(diag) + np.diag(offd, 1) + np.diag(offd, -1)

    # Matrice effective Newmark : K_eff = I + beta*dt²*K
    K_eff = np.eye(N) + beta * dt**2 * K

    u   = np.sin(np.pi * x)
    v   = np.zeros(N)
    a   = -K @ u

    snapshots = [(0.0, u.copy())]
    record_at = {int(0.5/dt), int(1.0/dt), int(T/dt)-1}

    for n in range(nt):
        u_pred = u + dt*v + dt**2*(0.5 - beta)*a
        v_pred = v + dt*(1 - gamma)*a
        a_new  = np.linalg.solve(K_eff, -K @ u_pred)
        u      = u_pred + beta*dt**2*a_new
        v      = v_pred + gamma*dt*a_new
        a      = a_new
        if n in record_at:
            snapshots.append(((n+1)*dt, u.copy()))

    return x, snapshots

x, snaps_w = ondes_newmark()
u_exact_ondes = lambda x, t: np.cos(np.pi * t) * np.sin(np.pi * x)

plt.figure(figsize=(9, 4))
colors = ['steelblue', 'orange', 'green', 'tomato']
for (t, u), col in zip(snaps_w, colors):
    plt.plot(x, u, '-', color=col, label=f't={t:.2f} (num.)')
    plt.plot(x, u_exact_ondes(x, t), '--', color=col, alpha=0.5)
plt.title('Équation des ondes — schéma de Newmark (β=1/4, γ=1/2)')
plt.xlabel('x'); plt.legend(fontsize=8); plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

t_f, u_f = snaps_w[-1]
err_ondes = np.max(np.abs(u_f - u_exact_ondes(x, t_f)))
print(f'Erreur max à t={t_f:.2f} : {err_ondes:.2e}')

## Tableau récapitulatif des schémas

| EDP | Schéma | Stabilité | Ordre en temps | Ordre en espace |
|-----|--------|-----------|---------------|-----------------|
| Poisson | Diff. finies centrées | — | — | $O(h^2)$ |
| Chaleur | Euler implicite | Inconditionnelle | $O(\Delta t)$ | $O(h^2)$ |
| Ondes   | Newmark $\beta=1/4$ | Inconditionnelle | $O(\Delta t^2)$ | $O(h^2)$ |

**Remarque** : l'Euler explicite pour la chaleur est conditionnellement stable ($r \le 1/2$).
L'Euler implicite élimine cette contrainte au prix d'une résolution de système linéaire à chaque pas.